# Lab 67: The Gaze Follower

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 8, *The Social Network*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-67.ipynb)

**What you will do:** count how many bystanders follow a friend's gaze when one person looks versus when several people look at once, compare the pattern with the book's prediction, and see where an AI vision model's own "attention" comes from.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 67 you and a friend tested **joint attention** in a crowded public space: your friend
suddenly stared at a fixed point, and you counted how many nearby strangers followed the
gaze — first with one person looking, then with two or three people looking at once. The book
predicts that adding more simultaneous gazers increases the number of bystanders who join in
"dramatically", because gaze-following is an automatic response you cannot easily suppress,
not a deliberate choice.

## Record your results

Log each trial: how many bystanders were close enough to notice, and how many of them
followed the gaze within a few seconds. Run at least two trials per condition if you can —
public settings are noisy, and a single trial can go either way by chance.

In [ ]:
# example data: replace with yours
trials = pd.DataFrame({
    "condition": ["One person looking", "One person looking",
                  "Three people looking", "Three people looking"],
    "trial": [1, 2, 1, 2],
    "bystanders_nearby": [10, 14, 12, 9],
    "bystanders_who_followed": [2, 3, 8, 6],
})
trials["pct_followed"] = (trials.bystanders_who_followed / trials.bystanders_nearby * 100).round(0)
display(trials)

trials.groupby("condition")["pct_followed"].mean().plot.bar(
    color=["steelblue", "firebrick"], figsize=(5, 3.5))
plt.ylabel("% of bystanders who followed the gaze"); plt.ylim(0, 100)
plt.xticks(rotation=0); plt.show()

## Compare

Lab 67 does not give an exact percentage to match — it predicts a direction: more
simultaneous gazers should pull in more followers. Check whether your own data shows that
direction.

In [ ]:
one = trials.loc[trials.condition == "One person looking", "pct_followed"].mean()
three = trials.loc[trials.condition == "Three people looking", "pct_followed"].mean()
print(f"One person looking: {one:.0f}% of bystanders followed")
print(f"Three people looking: {three:.0f}% of bystanders followed")
if three > one:
    print("This matches Lab 67's prediction — more simultaneous gazers pulled in more followers.")
else:
    print("This does not match the predicted direction. Public settings are noisy (passers-by")
    print("distracted, short observation windows), so a few trials can easily go either way.")

## The AI side

Your brain infers what someone finds interesting by tracking their eyes. **Grad-CAM**
(Selvaraju et al., 2017) is a rough artificial parallel: it highlights the pixels that most
influenced a convolutional network's classification decision, as if the network were
pointing at what it looked at. Below, a small pretrained image classifier looks at a photo,
and Grad-CAM draws a heatmap over the region that drove its guess.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from skimage import data
from torchvision import models, transforms

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.eval()

img = Image.fromarray(data.chelsea())  # a cat photo bundled with scikit-image
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = preprocess(img).unsqueeze(0)

activations, gradients = [], []
target_layer = model.layer4[-1]
target_layer.register_forward_hook(lambda m, i, o: activations.append(o))
target_layer.register_full_backward_hook(lambda m, gi, go: gradients.append(go[0]))

output = model(input_tensor)
class_idx = output.argmax(dim=1).item()
model.zero_grad()
output[0, class_idx].backward()

weights = gradients[0].mean(dim=(2, 3), keepdim=True)
cam = F.relu((weights * activations[0]).sum(dim=1)).squeeze()
cam = cam / (cam.max() + 1e-8)
cam_resized = F.interpolate(cam[None, None], size=(224, 224), mode="bilinear")[0, 0].detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img.resize((224, 224))); axes[0].set_title("Input image"); axes[0].axis("off")
axes[1].imshow(img.resize((224, 224)))
axes[1].imshow(cam_resized, cmap="jet", alpha=0.5)
axes[1].set_title("Grad-CAM: where the network 'looked'"); axes[1].axis("off")
plt.tight_layout(); plt.show()
print("Predicted ImageNet class index:", class_idx)

Compare where the heat is concentrated with the object the network actually named. The
analogy has real limits: the network's "attention" is a fixed byproduct of gradients
flowing back through convolutional filters trained once and never updated again — it is not
a live, moving fixation driven by curiosity, and it cannot be redirected by someone else's
gaze the way yours constantly is.

## Pool the class data

Pool the class's bystander counts to see whether the effect of adding more gazers holds up
across many different public spaces and times of day, not just your own two trials.

In [ ]:
import io
csv_text = """id,condition,pct_followed
P01,one_looker,15
P02,one_looker,25
P03,three_lookers,55
P04,three_lookers,70
P05,one_looker,10
P06,three_lookers,60
P07,one_looker,20
P08,three_lookers,45
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
means = class_df.groupby("condition").pct_followed.mean()

diffs = []
one_vals = class_df.loc[class_df.condition == "one_looker", "pct_followed"].values
three_vals = class_df.loc[class_df.condition == "three_lookers", "pct_followed"].values
for _ in range(5000):
    diffs.append(np.random.choice(three_vals, len(three_vals), replace=True).mean()
                  - np.random.choice(one_vals, len(one_vals), replace=True).mean())
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"Class mean, one looker: {means['one_looker']:.0f}%, three lookers: {means['three_lookers']:.0f}%")
print(f"Bootstrap 95% interval for the difference: {lo:.0f} to {hi:.0f} percentage points")
means.plot.bar(color=["steelblue", "firebrick"], figsize=(5, 3.5))
plt.ylabel("% followed"); plt.show()

## Questions to think about

1. Your friend's gaze in Step 1 was posed for the demonstration. Try noticing a stranger's spontaneous glance at something instead — does an unplanned gaze pull your own attention just as strongly?
2. The gaze-cueing paradigm (Friesen and Kingstone, 1998) uses a screen and a stopwatch instead of counting bystanders. What would you gain by measuring reaction time in milliseconds rather than counting how many people turned their heads?
3. Joint attention is one of the earliest markers used to screen for autism spectrum conditions. Given how automatic and hard to suppress the effect is in adults, why might its absence in infancy be such a informative early sign?
4. The Grad-CAM heatmap shows where a fixed set of trained filters "looked" for one static image. What would a network need in order to follow a moving point of interest the way your eyes follow a friend's gaze in real time?

## Challenge

Run the experiment in a setting where the "looker" has no idea they are being watched — for
example, notice a stranger glancing up at something on a train platform or in a shop, and see
whether nearby people follow their gaze the same way they followed your friend's posed stare.
A timed, on-screen version of the classic gaze-cueing paradigm would let you measure the
reaction-time advantage directly, in milliseconds; a timed version will be added.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-67.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")